# Reading floor plans: every change, measured

The software has to look at an estate agent's floor plan and work out where the rooms are.
It has not been doing that well. This notebook tries **six changes, one at a time**, and
after every single one it re-runs all 25 test plans and tells you whether things got better
or worse.

Nothing is taken on trust. Every step ends with a number and a set of pictures.

**At the end** you get one table, one chart, and a picture of every flat showing our reading,
the other model's reading, and the two combined, side by side.

---

### How we score

Every room the software finds is a shape drawn on the plan. We walk around the edge of that
shape and ask, at every step along it: **is there a wall here?**

- A room read correctly has its edge on the walls the whole way round → **near 100%**
- A room whose edge cuts across open floor — because it stopped at the kitchen units, or
  followed the curve of a door — scores **low**

We call that number the **wall match**. One per plan; we report the middle one across all
25 so a single disastrous plan cannot drag the headline around.

**It has a known blind spot, and it is worth knowing before you read any number below.**
A room that has ballooned out *past* the building still has its edge running along a wall —
the outside of one — so it can score well while being obviously wrong to the eye. We tried
three ways to catch that automatically and none of them worked reliably yet, so **the
pictures are the arbiter, not the score.** Every step prints both.

---

### Before you start

**Attach a GPU.** In the sidebar, open the kernel settings and pick a GPU — a **T4** is
plenty, and an **A10G** is quicker. About 45 minutes end to end, most of it one compile.

**Files here are temporary.** Modal's container is wiped when the kernel stops, so the last
cell writes a zip you can pull down from the file browser in the sidebar. If you want the
work to survive on its own, mount a Volume and point `ROOT` at it.

# Part 1 — Setup

Plumbing. Run these five cells and don't read them; the interesting part starts at Part 2.

### 1.1 · Is there a GPU?

In [ ]:
import subprocess

import torch

try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("no nvidia-smi on this runtime")
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU, then rerun"

### 1.2 · Get the two codebases

Ours, and the room-finding model's.

In [ ]:
import os
import subprocess
from pathlib import Path

# Everything this notebook makes lives here. Point it at /mnt/<your-volume> if you
# have mounted one and want the results to outlive the kernel.
ROOT = Path("/root/plan-reading")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

R2S = ROOT / "Raster2Seq"
OURS = ROOT / "visit-it"


def clone(url, dest):
    if dest.exists():
        print(f"{dest.name}: already here")
        return
    r = subprocess.run(["git", "clone", "--depth", "1", url, str(dest)],
                       capture_output=True, text=True)
    print(f"{dest.name}: {'cloned' if not r.returncode else 'FAILED'}")
    if r.returncode:
        print(r.stderr[-600:])


clone("https://github.com/Cornell-VAILab/Raster2Seq.git", R2S)
clone("https://github.com/romainbigare/visit-it.git", OURS)
os.chdir(R2S)

### 1.3 · Install what is missing — carefully

Colab ships NumPy, OpenCV and Matplotlib as one set, all built against each other. Moving
any of them breaks the rest, so we pin NumPy where it already is and install only what is
genuinely absent.

In [ ]:
import shutil
import subprocess
import sys

import numpy

# Hold NumPy exactly where the image put it. OpenCV, SciPy and Matplotlib are all
# built against a particular NumPy, and moving it breaks every one of them.
PIN = f"numpy=={numpy.__version__}"
# Read off what the code actually imports, not guessed at. The vendored copy of
# detectron2 drags in a long tail of small packages -- cloudpickle, hydra, iopath,
# tabulate, termcolor, black -- that Colab happened to ship and a clean image does
# not. The preflight in 1.6 catches anything still missing.
NEEDED = ["opencv-python-headless", "scikit-image", "scipy", "shapely", "plotly",
          "imageio", "descartes", "omegaconf", "fvcore", "pycocotools",
          "segmentation-models-pytorch", "safetensors", "pytesseract", "timm",
          "cloudpickle", "hydra-core", "iopath", "tabulate", "termcolor", "black",
          "pyyaml", "yacs", "portalocker"]


def pip_install(*packages):
    """Install with NumPy held still. Use this for everything in this notebook —
    one unpinned install anywhere is enough to break the binary packages."""
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", PIN, *packages],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-1500:], r.stderr[-1500:])
        raise SystemExit(f"pip could not install {packages} alongside {PIN}")


print(f"holding {PIN}")
pip_install(*NEEDED)

# Tesseract is a program, not a Python package, and our own reading needs it to
# find the room names printed on the plan.
if shutil.which("tesseract") is None:
    subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "tesseract-ocr"],
                   capture_output=True)
HAVE_TESSERACT = shutil.which("tesseract") is not None
print("tesseract:", "installed" if HAVE_TESSERACT else
      "NOT AVAILABLE — step 0 will find fewer rooms than it should")

check = subprocess.run(
    [sys.executable, "-c",
     "import numpy, cv2, matplotlib, shapely, plotly, imageio, torch, skimage;"
     "print(numpy.__version__, cv2.__version__, matplotlib.__version__)"],
    capture_output=True, text=True)
if check.returncode:
    print(check.stderr[-1200:])
    raise SystemExit("something broke — restart the kernel and run from the top")
print("numpy / opencv / matplotlib:", check.stdout.strip(), "— all fine")

### 1.4 · Make the room-finder run, and build it

Raster2Seq was written for early-2024 libraries and four things have moved since. Each has
an exact modern equivalent, so these are renames, not rewrites. Then it compiles its two
GPU pieces — this is the slow cell, around five minutes.

In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

import torch

REPO = R2S

CMAP_OLD = "from matplotlib.cm import get_cmap"
CMAP_MARK = "# patched: matplotlib >= 3.9"
CMAP_NEW = f"""try:
    {CMAP_OLD}
except ImportError:                      {CMAP_MARK} removed it
    from matplotlib import colormaps

    def get_cmap(name=None, lut=None):
        return colormaps[name]"""


def modernise(root: Path) -> dict:
    """Four renames. Each is guarded against matching its own output, so running
    this twice changes nothing the second time."""
    hits = {}
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.suffix in (".cu", ".cuh", ".cpp", ".h", ".hpp"):
            src = original = path.read_text()
            src, a = re.subn(r"(AT_DISPATCH_\w+\(\s*)(\w+)\.type\(\)",
                             r"\1\2.scalar_type()", src)          # Tensor::type() is gone
            src, b = re.subn(r"(\w+)\.type\(\)\.is_cuda\(\)", r"\1.is_cuda()", src)
            n = a + b
        elif path.suffix == ".py":
            src = original = path.read_text()
            n = 0
            if CMAP_MARK not in src and CMAP_OLD in src:          # matplotlib 3.9
                src = src.replace(CMAP_OLD, CMAP_NEW, 1)
                n += 1
            src, c = re.subn(                                     # pytorch 2.6
                r"torch\.load\(([^)]*?map_location=[^)]*?)\)",
                lambda m: (m.group(0) if "weights_only" in m.group(1)
                           else f"torch.load({m.group(1)}, weights_only=False)"),
                src)
            n += c
        else:
            continue
        if src != original:
            path.write_text(src)
            hits[str(path.relative_to(root))] = n
    return hits


edits = modernise(REPO)
print(f"patched {len(edits)} files" if edits else "nothing left to patch")

major, minor = torch.cuda.get_device_capability()
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
print(f"\nbuilding for {torch.cuda.get_device_name(0)} — about five minutes\n")


def build(where: Path) -> bool:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--no-build-isolation", "."],
                       cwd=where, capture_output=True, text=True, env=os.environ)
    print(f"{where.name}: {'built' if not r.returncode else 'FAILED'}")
    if r.returncode:
        for line in (r.stdout + r.stderr).splitlines()[-10:]:
            print("   ", line)
    return not r.returncode


build(REPO / "models" / "ops")
if not build(REPO / "diff_ras"):
    print("  (the second one is only used by a feature we switch off — this is fine)")

### 1.5 · Confirm the room-finder works

If the GPU piece did not build, there is a slower pure-Python version of the same
calculation. Identical answers, several times slower. Either is fine here.

In [ ]:
import importlib
import importlib.util
import site
import sys
from pathlib import Path

import torch

REPO = R2S
sys.path.insert(0, str(REPO))
FUNC = REPO / "models" / "ops" / "functions" / "ms_deform_attn_func.py"
UPSTREAM = "import MultiScaleDeformableAttention as MSDA"

SHIM = """
try:
    import MultiScaleDeformableAttention as MSDA
except ImportError:
    # The GPU piece did not build. The same calculation in plain PyTorch is
    # further down this file, so route through it. Inference only.
    class _PurePythonMSDA:
        @staticmethod
        def ms_deform_attn_forward(value, value_spatial_shapes, value_level_start_index,
                                   sampling_locations, attention_weights, im2col_step):
            return ms_deform_attn_core_pytorch(
                value, value_spatial_shapes, sampling_locations, attention_weights)

        @staticmethod
        def ms_deform_attn_backward(*_args, **_kwargs):
            raise RuntimeError("inference only without the compiled extension")

    MSDA = _PurePythonMSDA()
"""


def find_extension():
    """A package installed while this kernel was already running is invisible to it
    until the import caches are dropped. Worth trying before concluding it failed."""
    importlib.invalidate_caches()
    try:
        return importlib.import_module("MultiScaleDeformableAttention")
    except ImportError:
        pass
    site.main()
    roots = list(site.getsitepackages())
    for root in roots:
        for egg in Path(root).glob("MultiScaleDeformableAttention*.egg"):
            if str(egg) not in sys.path:
                sys.path.insert(0, str(egg))
    importlib.invalidate_caches()
    try:
        return importlib.import_module("MultiScaleDeformableAttention")
    except ImportError:
        return None


if find_extension() is not None:
    SLOW_PATH = False
    print("room-finder: fast GPU version")
else:
    src = FUNC.read_text()
    if "_PurePythonMSDA" not in src:
        FUNC.write_text(src.replace(UPSTREAM, SHIM.strip(), 1))
    for name in [m for m in sys.modules if "ms_deform_attn" in m]:
        del sys.modules[name]
    SLOW_PATH = True
    print("room-finder: slower plain-PyTorch version (same answers)")

spec = importlib.util.spec_from_file_location("_check", FUNC)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
shapes = torch.as_tensor([[8, 6], [4, 3]], dtype=torch.long, device="cuda")
sizes = shapes[:, 0] * shapes[:, 1]
with torch.no_grad():
    out = mod.MSDeformAttnFunction.apply(
        torch.randn(2, int(sizes.sum()), 8, 32, device="cuda"), shapes,
        torch.cat([sizes.new_zeros(1), sizes.cumsum(0)[:-1]]),
        torch.rand(2, 7, 8, 2, 4, 2, device="cuda"),
        torch.rand(2, 7, 8, 2, 4, device="cuda"), 64)
assert tuple(out.shape) == (2, 7, 256) and torch.isfinite(out).all()
print("   checked — it computes the right thing")

# The model's own import chain, exercised before we spend five minutes discovering
# it at the first run. A missing module names itself in the error, so install it and
# try again rather than making you paste the traceback back to me.
PIP_NAME = {"cv2": "opencv-python-headless", "skimage": "scikit-image",
            "PIL": "pillow", "sklearn": "scikit-learn", "yaml": "pyyaml",
            "hydra": "hydra-core", "shapely": "shapely", "pycocotools": "pycocotools",
            "google": "protobuf", "cairosvg": "cairosvg", "drawsvg": "drawsvg",
            "svgpathtools": "svgpathtools", "svgwrite": "svgwrite",
            "plyfile": "plyfile", "webcolors": "webcolors"}


def preflight(rounds=12):
    tried = set()
    for _ in range(rounds):
        r = subprocess.run([sys.executable, "-c", "import predict"],
                           cwd=str(R2S), capture_output=True, text=True)
        if r.returncode == 0:
            return True
        err = r.stderr.strip().splitlines()
        missing = next((ln.split("'")[1] for ln in reversed(err)
                        if "ModuleNotFoundError: No module named" in ln), None)
        if not missing:
            print("   the model cannot start, and it is not a missing package:")
            print("\n".join(err[-12:]))
            return False
        pkg = PIP_NAME.get(missing, missing)
        if missing in tried:
            # Installing it did not make it importable, so the name we guessed is
            # wrong. Say which, rather than reinstalling it until the loop runs out.
            print(f"   installed {pkg}, but '{missing}' still will not import.")
            print(f"   Add the right pip name for '{missing}' to PIP_NAME above "
                  f"and re-run this cell.")
            return False
        tried.add(missing)
        print(f"   missing {missing} → installing {pkg}")
        try:
            pip_install(pkg)
        except SystemExit:
            print(f"   could not install {pkg} — stopping here")
            return False
    print(f"   still missing things after {rounds} rounds — stopping")
    return False


print()
print("checking the model can start")
READY = preflight()
print("   ready" if READY else "   NOT ready — the steps below will report nothing")

### 1.6 · The 25 test plans

Downloaded straight from the addresses recorded in our own test set, so nothing needs
uploading and nothing needs to exist on your machine.

In [ ]:
import json
import ssl
import urllib.request
from pathlib import Path

PLANS = ROOT / "plans" / "original"
PLANS.mkdir(parents=True, exist_ok=True)
golden = json.loads((OURS / "data" / "golden" / "golden_set.json").read_text())

ctx = ssl.create_default_context()
missing = []
for listing in golden["listings"]:
    plans = listing.get("floorplans") or []
    if not plans:
        missing.append(listing["listing_id"])
        continue
    dest = PLANS / f"{listing['listing_id']}.png"
    if dest.exists():
        continue
    try:
        req = urllib.request.Request(plans[0]["url"], headers={"User-Agent": "Mozilla/5.0"})
        dest.write_bytes(urllib.request.urlopen(req, context=ctx, timeout=60).read())
    except Exception as exc:
        print("  could not fetch", listing["listing_id"], exc)
        missing.append(listing["listing_id"])

IDS = sorted(p.stem for p in PLANS.glob("*.png"))
print(f"{len(IDS)} plans ready" + (f"  ·  {len(missing)} listings have no plan" if missing else ""))

### 1.7 · The wall map, and the score

One model in this notebook does nothing but say, for every pixel, whether it is a wall.
That is what we score against — because "is this edge on *some* line?" cannot tell a wall
from a kitchen cabinet, and cabinets are exactly what has been going wrong.

This cell downloads that model and defines the score. Read the two short functions if you
like; you do not need to.

In [ ]:
import sys
import urllib.request
import warnings
from pathlib import Path

import numpy as np
from PIL import Image

sys.path.insert(0, str(OURS))
warnings.filterwarnings("ignore")

from pipeline.floorplan import ocr as ocr_mod
from pipeline.floorplan import preprocess, vectorise, wallnet

(OURS / "models").mkdir(parents=True, exist_ok=True)
wallnet.MODEL_PATH = OURS / "models" / "plan_walls.safetensors"
if not wallnet.MODEL_PATH.exists():
    urllib.request.urlretrieve(wallnet.MODEL_URL, wallnet.MODEL_PATH)
assert wallnet.available(), "the wall model did not load"

# Two pictures of every plan, and a wall map for each.
#
#   "original"  — the file as the agent published it. The room-finder reads these.
#   "geometry"  — our own pipeline straightens and shrinks the plan before working on
#                 it, so its answers are in *those* pixels.
#
# An answer has to be scored against the wall map of the picture it was drawn on. Score
# it against the other one and every edge lands a couple of percent out, which looks
# exactly like the software being wrong when it is not.
WALLS, GEOM_IMG, GEOM_SIZE = {}, {}, {}
print("building a wall map for each plan (about a minute)")
for lid in IDS:
    rgb = np.array(Image.open(PLANS / f"{lid}.png").convert("RGB"))
    ink, _ = preprocess.ink_mask(rgb)
    WALLS[(lid, "original")] = wallnet.barrier(rgb, ink, [])

    pi = preprocess.prepare(PLANS / f"{lid}.png")
    text = ocr_mod.read(pi.rgb)
    WALLS[(lid, "geometry")] = wallnet.barrier(pi.rgb, pi.ink, text.words, pi.wall_half_px)
    GEOM_IMG[lid] = pi.rgb
    GEOM_SIZE[lid] = (pi.ink.shape[1], pi.ink.shape[0])
print("done")


def wall_match(reading, space="original"):
    """Share of every room's edge that lies on a wall. One number per plan."""
    per_plan = {}
    for lid, rec in reading.items():
        ref = WALLS.get((lid, space))
        if ref is None or not ref.any():
            continue
        polys = [r["polygon_px"] for r in rec["rooms"] if len(r["polygon_px"]) >= 3]
        scores = wallnet.outline_on_wall(polys, ref)
        if scores:
            per_plan[lid] = float(np.median(scores))
    return per_plan


LADDER = []


def record(name, reading, space, note):
    """Score a reading, add it to the ladder, and say how it did."""
    per_plan = wall_match(reading, space)
    if not per_plan:
        print(f"{name}: produced nothing to score")
        return None
    score = float(np.median(list(per_plan.values())))
    rooms = sum(len(r["rooms"]) for r in reading.values())
    entry = {"name": name, "score": score, "plans": len(per_plan), "rooms": rooms,
             "note": note, "space": space, "per_plan": per_plan}
    LADDER.append(entry)
    prev = LADDER[-2]["score"] if len(LADDER) > 1 else None
    best = max(e["score"] for e in LADDER[:-1]) if len(LADDER) > 1 else None
    line = f"  wall match  {score:.0%}   ·  {rooms} rooms found across {len(per_plan)} plans"
    print(line)
    if prev is not None:
        arrow = "better" if score > prev + 0.005 else ("worse" if score < prev - 0.005 else "no change")
        print(f"  previous step was {prev:.0%}  →  {arrow}")
    if best is not None and score > best + 0.005:
        print(f"  best so far")
    return entry

### 1.8 · Running the room-finder

One function, used by every step below. It writes the plans somewhere, runs the model over
them, and hands back the rooms it found in the plan's own pixels.

In [ ]:
import json
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import numpy as np
from PIL import Image

# Every checkpoint publishes the settings it was trained with, so read them rather
# than hand-writing flags. Hand-writing them is how the "sharper model" step failed
# first time round: it was run at 32 coordinate bins with 12 room types when that
# checkpoint wants 32 bins and 13, and against the 256-pixel weights while asking
# for a 512-pixel model.
HUB = "https://huggingface.co/haopt/Raster2Seq/resolve/main"

# What the room types are called, per training set.
CC5K_NAMES = {0: "Outdoor", 1: "Kitchen", 2: "Living Room", 3: "Bed Room", 4: "Bath",
              5: "Entry", 6: "Storage", 7: "Garage", 8: "Undefined"}
CC5K_APERTURES = {9, 10}                       # window and door, kept out of the rooms
R2G_NAMES = {0: "unknown", 1: "living room", 2: "kitchen", 3: "bedroom", 4: "bathroom",
             5: "restroom", 6: "balcony", 7: "closet", 8: "corridor",
             9: "washing room", 10: "PS", 11: "outside"}
R2G_APERTURES = set()                          # this one marks no doors or windows

_CONFIGS = {}


def hub_subfolder(key):
    """Where that checkpoint lives on the hub.

    The name you pass and the folder it sits in are not always the same -- the
    512-pixel one is called `raster2graph-512` and stored under `Raster2Graph-512`
    -- so read the repository's own table rather than keeping a second copy of it
    here that can drift.
    """
    try:
        sys.path.insert(0, str(R2S))
        import raster2seq_hub
        canonical = raster2seq_hub.normalize_checkpoint_name(key)
        return canonical, str(raster2seq_hub.CHECKPOINTS[canonical]["subfolder"])
    except Exception:
        return key, key


def checkpoint_config(key):
    """The settings that checkpoint was trained with, straight from the hub."""
    if key not in _CONFIGS:
        canonical, folder = hub_subfolder(key)
        with urllib.request.urlopen(f"{HUB}/{folder}/config.json", timeout=60) as r:
            _CONFIGS[key] = json.load(r)
        _CONFIGS[key]["_alias"] = canonical
    return _CONFIGS[key]


def flags_from(config):
    """Turn a checkpoint's own inference_args into command-line flags."""
    args = dict(config["inference_args"])
    for drop in ("dataset_root", "eval_set", "output_dir"):
        args.pop(drop, None)
    out = []
    for k, v in args.items():
        if v is True:
            out.append(f"--{k}")
        elif v is False or v is None:
            continue
        else:
            out += [f"--{k}", str(v)]
    return out, int(args.get("image_size", 256)), args.get("dataset_name", "cubicasa")


def undo_letterbox(poly, src_w, src_h, size):
    """The model works on a small square copy with grey bars; put the coordinates back."""
    scale = min(size / src_h, size / src_w)
    new_h, new_w = int(src_h * scale), int(src_w * scale)
    left, top = (size - new_w) // 2, (size - new_h) // 2
    p = np.asarray(poly, dtype=float).reshape(-1, 2)
    return np.stack([(p[:, 0] - left) / scale, (p[:, 1] - top) / scale], axis=1)


def find_rooms(images_dir, tag, checkpoint="cubicasa5k"):
    """Run the room-finder over a directory of plans, using that checkpoint's own
    settings. Returns {listing: {"rooms": [...]}} in each plan's own pixels."""
    if not READY:
        print("  skipped: the room-finder could not start — see the preflight in 1.5")
        return {}
    try:
        config = checkpoint_config(checkpoint)
    except Exception as exc:
        print(f"  could not read the settings for '{checkpoint}': {exc}")
        return {}
    flags, size, dataset = flags_from(config)
    names = R2G_NAMES if dataset == "r2g" else CC5K_NAMES
    apertures = R2G_APERTURES if dataset == "r2g" else CC5K_APERTURES
    print(f"  {config['name']}  ·  {size}px  ·  "
          f"scores {config['metrics']['room_f1']} on its own test set")

    out_dir = ROOT / "runs" / tag
    if out_dir.exists():
        shutil.rmtree(out_dir)
    t0 = time.time()
    cmd = ["python", "predict.py", f"--dataset_root={images_dir}",
           f"--output_dir={out_dir}", f"--checkpoint=hf:{config['_alias']}", *flags]
    r = subprocess.run(cmd, cwd=str(R2S), capture_output=True, text=True)
    if r.returncode:
        print(f"  the model failed on '{tag}':")
        for line in (r.stdout + r.stderr).splitlines()[-8:]:
            print("     ", line)
        return {}

    reading = {}
    for jf in sorted(out_dir.rglob("jsons/*.json")):
        lid = jf.stem
        base = lid.partition("__")[0]
        src = Image.open(PLANS / f"{base}.png") if (PLANS / f"{base}.png").exists() \
            else Image.open(Path(images_dir) / f"{lid}.png")
        rooms = []
        for inst in json.loads(jf.read_text()):
            cid = inst["category_id"]
            if cid in apertures or cid not in names:
                continue
            rooms.append({"polygon_px": undo_letterbox(inst["segmentation"],
                                                       src.width, src.height, size).tolist(),
                          "label": names[cid]})
        reading[lid] = {"rooms": rooms}
    print(f"  ran in {time.time() - t0:.0f}s")
    return reading

---

# Part 2 — The ladder

Six changes, one per section. Each one ends with a score and pictures.

## Step 0 · What we have today

The starting point. Our own software: a model marks every wall, then each room is grown
outwards from a starting point until it hits one. The starting points come from the room
names printed on the plan, which is the weak part — a room with no printed name has no
starting point, and a caption reading "KITCHEN / LIVING ROOM" produces two starting points
in what is really one room.

In [ ]:
ours = {}
for lid in IDS:
    try:
        pi = preprocess.prepare(PLANS / f"{lid}.png")
        text = ocr_mod.read(pi.rgb)
        v = vectorise.segment(pi, text)
    except Exception as exc:
        print(f"  {lid}: {type(exc).__name__}: {exc}")
        continue
    ours[lid] = {"rooms": [{"polygon_px": r.polygon_px, "label": r.label or ""}
                           for r in v.rooms]}

record("What we have today", ours, "geometry",
       "our software: wall model, rooms grown from the printed room names")

## Step 1 · A second opinion

A different model — **Raster2Seq** — which does not look for walls at all. It looks at the
whole plan and says directly: here is a kitchen, here is a bedroom, here is a bathroom. It
never reads the printed names, so a room with no label is no harder for it than one with.

Run on the plans exactly as published.

In [ ]:
raw = find_rooms(PLANS, "raw", checkpoint="cubicasa5k")
record("Second opinion, plans as published", raw, "original",
       "Raster2Seq on the original files")

## Step 2 · Give it a cleaner picture

This model learned on plans drawn as black lines on white paper, with no text. Ours are
often tinted, colour-filled, and covered in room names and dimensions.

So we hand it a tidied copy: **the page levelled to white**, and **every word painted out**.
Nothing about the model changes — only what it is looking at.

In [ ]:
import cv2
import pytesseract

CLEAN = ROOT / "plans" / "cleaned"
CLEAN.mkdir(parents=True, exist_ok=True)


def level_to_white(rgb):
    """Whatever colour the page is, make it white; whatever the darkest ink is, black."""
    lum = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    page = int(np.argmax(np.bincount(lum.ravel(), minlength=256)))
    if page < 110:                       # white lines on a dark page: flip first
        lum, page = 255 - lum, 255 - page
    floor = float(np.percentile(lum, 1))
    if page - floor < 20:
        return cv2.cvtColor(lum, cv2.COLOR_GRAY2RGB)
    out = np.clip((lum.astype(np.float32) - floor) * (255.0 / (page - floor)), 0, 255)
    return cv2.cvtColor(out.astype(np.uint8), cv2.COLOR_GRAY2RGB)


def paint_out_words(rgb):
    data = pytesseract.image_to_data(rgb, output_type=pytesseract.Output.DICT)
    out, h, w = rgb.copy(), *rgb.shape[:2]
    for i, conf in enumerate(data["conf"]):
        if float(conf) < 40 or not data["text"][i].strip():
            continue
        x, y = data["left"][i] - 2, data["top"][i] - 2
        out[max(0, y):min(h, y + data["height"][i] + 4),
            max(0, x):min(w, x + data["width"][i] + 4)] = 255
    return out


for lid in IDS:
    dest = CLEAN / f"{lid}.png"
    if not dest.exists():
        rgb = np.array(Image.open(PLANS / f"{lid}.png").convert("RGB"))
        Image.fromarray(paint_out_words(level_to_white(rgb))).save(dest)

cleaned = find_rooms(CLEAN, "cleaned")
record("Second opinion, cleaned-up picture", cleaned, "original",
       "page levelled to white, all text painted out")

In [ ]:
# See what it is now looking at
import matplotlib.pyplot as plt

for lid in IDS[:3]:
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    for a, d, t in zip(ax, (PLANS, CLEAN), ("as published", "cleaned up for the model")):
        a.imshow(Image.open(d / f"{lid}.png")); a.axis("off"); a.set_title(t, fontsize=10)
    plt.tight_layout(); plt.show()

## Step 3 · Try the sharper model

The authors publish several trained models. The one we have used so far works at
**256 pixels**; another works at **512**, so the corners it reports can land twice as
precisely. It was trained on a different collection of plans, with a different list of
room types, which may suit ours better or worse.

Each model comes with a record of the settings it was trained with, and this notebook
reads that record rather than assuming — the number of coordinate bins, the number of room
types and the working resolution all differ between them, and getting any one of them
wrong means the model refuses to load at all.

In [ ]:
# The authors also publish this one trained at 512 pixels instead of 256, on a
# different set of plans. Its own settings come with it, so nothing here is guessed.
sharper = find_rooms(CLEAN, "sharper", checkpoint="raster2graph-512")
if sharper:
    record("The sharper 512-pixel model", sharper, "original",
           "trained at twice the resolution, on a different set of plans")
else:
    print("  this step did not produce anything — carrying on with the best so far")

## Step 4 · Show it the plan in pieces

The model shrinks the whole plan to a small square before looking at it, so a cupboard or a
WC can end up only a few pixels across and disappear.

So we cut each plan into four overlapping quarters, run the model on each at full size, and
put the answers back together — keeping whichever version of an overlapping room covers
more floor.

This is the least likely of the six to work: a quarter of a flat is not a flat, and the
model was trained on whole ones.

In [ ]:
from itertools import product

TILES = ROOT / "plans" / "tiles"


def cut_into_tiles(overlap=0.25):
    if TILES.exists():
        shutil.rmtree(TILES)
    TILES.mkdir(parents=True)
    boxes = {}
    for lid in IDS:
        im = Image.open(CLEAN / f"{lid}.png").convert("RGB")
        w, h = im.size
        tw, th = int(w * (0.5 + overlap / 2)), int(h * (0.5 + overlap / 2))
        for i, (cx, cy) in enumerate(product((0, w - tw), (0, h - th))):
            name = f"{lid}__{i}"
            im.crop((cx, cy, cx + tw, cy + th)).save(TILES / f"{name}.png")
            boxes[name] = (cx, cy, tw, th)
    return boxes


def merge_tiles(tile_reading, boxes):
    """Put every tile's rooms back on the full plan, then drop the duplicates."""
    from shapely.geometry import Polygon
    merged = {}
    for name, rec in tile_reading.items():
        lid, _, _idx = name.partition("__")
        cx, cy, _tw, _th = boxes[name]
        for room in rec["rooms"]:
            p = np.asarray(room["polygon_px"], float) + [cx, cy]
            merged.setdefault(lid, []).append({"polygon_px": p.tolist(),
                                               "label": room["label"]})
    out = {}
    for lid, rooms in merged.items():
        polys = []
        for room in sorted(rooms, key=lambda r: -_area(r["polygon_px"])):
            try:
                g = Polygon(room["polygon_px"]).buffer(0)
            except Exception:
                continue
            if g.is_empty or g.area <= 0:
                continue
            # a room already covered by a bigger one from another tile is the same room
            if any(g.intersection(k).area > 0.5 * g.area for k in polys):
                continue
            polys.append(g)
            out.setdefault(lid, {"rooms": []})["rooms"].append(room)
    return out


def _area(poly):
    p = np.asarray(poly, float)
    return abs(float(np.dot(p[:, 0], np.roll(p[:, 1], 1)) -
                     np.dot(p[:, 1], np.roll(p[:, 0], 1))) / 2)


tiled = {}
try:
    boxes = cut_into_tiles()
    # find_rooms names results after the file, and tiles are named "<listing>__<n>"
    tile_reading = find_rooms(TILES, "tiles")
    tiled = merge_tiles(tile_reading, boxes)
except Exception as exc:
    print("  did not run:", type(exc).__name__, exc)

if tiled:
    record("Shown the plan in four pieces", tiled, "original",
           "four overlapping quarters, answers merged")
else:
    print("  this step did not produce anything — carrying on with the best so far")

## Step 5 · Ask several times, keep what agrees

Show the model the plan three ways — as published, cleaned up, and cleaned up but rotated a
little — and keep the rooms that turn up in more than one. A room only one version finds is
usually a mistake; a room all three find is usually real.

In [ ]:
from shapely.geometry import Polygon

ROT = ROOT / "plans" / "rotated"
ROT.mkdir(parents=True, exist_ok=True)
for lid in IDS:
    dest = ROT / f"{lid}.png"
    if not dest.exists():
        Image.open(CLEAN / f"{lid}.png").rotate(2, expand=True, fillcolor=(255, 255, 255)).save(dest)

votes = {}
try:
    rotated = find_rooms(ROT, "rotated")
    for source in (raw, cleaned, rotated):
        for lid, rec in source.items():
            votes.setdefault(lid, []).extend(rec["rooms"])
except Exception as exc:
    print("  did not run:", exc)

consensus = {}
for lid, rooms in votes.items():
    shapes, kept = [], []
    for room in sorted(rooms, key=lambda r: -_area(r["polygon_px"])):
        try:
            g = Polygon(room["polygon_px"]).buffer(0)
        except Exception:
            continue
        if g.is_empty or g.area <= 0:
            continue
        hits = [i for i, k in enumerate(shapes)
                if g.intersection(k).area > 0.5 * min(g.area, k.area)]
        if hits:
            continue                       # same room, already have the larger version
        shapes.append(g)
        kept.append(room)
    if kept:
        consensus[lid] = {"rooms": kept}

if consensus:
    record("Asked three ways, kept the agreement", consensus, "original",
           "as published + cleaned + rotated, overlapping rooms merged")

## Step 6 · Put the two together

Here is the thing worth understanding, and it is the whole point of the notebook.

**The two models are good at opposite halves of the job.**

|  | the room-finder | the wall model |
|---|---|---|
| *which* rooms exist, and what they are | **it knows** | it guesses from the printed names |
| an open-plan kitchen-and-living-room | **one room** | split in two |
| a WC or cupboard with no printed name | **found** | usually missed |
| *where* the wall actually is | roughly — see below | **exactly** |

The room-finder shrinks the plan to a small square and rounds every corner to a coarse
grid — about **30 cm on a real flat**. That is baked into how it was trained; it is not a
setting anyone can turn up. So its rooms are the right rooms in roughly the right places,
with corners that miss the walls.

**So use each for what it is good at.** Take the room-finder's rooms as *starting points*,
and grow each one outwards until it hits a wall on the wall model's map. The rooms come
from the model that knows which rooms exist; the edges come from the model that knows where
walls are.

We already own the growing-outwards part — it is the same machinery Step 0 uses. The only
thing changing is where the starting points come from.

In [ ]:
# Whichever of steps 1-5 scored best is what we grow.
candidates = [e for e in LADDER if e["space"] == "original"]
best_finder = max(candidates, key=lambda e: e["score"]) if candidates else None
SOURCES = {"Second opinion, plans as published": raw,
           "Second opinion, cleaned-up picture": cleaned,
           "The sharper 512-pixel model": sharper,
           "Shown the plan in four pieces": tiled,
           "Asked three ways, kept the agreement": consensus}
if best_finder:
    seed_reading = SOURCES.get(best_finder["name"]) or cleaned
    print(f'growing the rooms from: {best_finder["name"]} ({best_finder["score"]:.0%})')
else:
    seed_reading = cleaned
    print("no room-finder step scored, growing the cleaned-up run")

joined = {}
for lid in IDS:
    rec = seed_reading.get(lid)
    if not rec or not rec["rooms"]:
        continue
    pi = preprocess.prepare(PLANS / f"{lid}.png")
    text = ocr_mod.read(pi.rgb)

    # The room-finder worked on the original file; our wall map is of the
    # straightened, shrunk copy. Move the rooms across before growing them.
    ow, oh = Image.open(PLANS / f"{lid}.png").size
    gw, gh = GEOM_SIZE[lid]
    seeds = [np.asarray(r["polygon_px"], float) * [gw / ow, gh / oh]
             for r in rec["rooms"] if len(r["polygon_px"]) >= 3]
    names = [r["label"] for r in rec["rooms"] if len(r["polygon_px"]) >= 3]

    grown = vectorise.segment_from_room_seeds(pi, text, seeds,
                                              mask=WALLS[(lid, "geometry")])
    if grown is None:
        continue
    joined[lid] = {"rooms": [{"polygon_px": r.polygon_px,
                              "label": names[i] if i < len(names) else (r.label or "")}
                             for i, r in enumerate(grown.rooms)]}

record("Both models together", joined, "geometry",
       "rooms from the room-finder, edges from the wall model")

---

# Part 3 — The report

Everything above, in one place.

### The table

In [ ]:
best = max(LADDER, key=lambda e: e["score"])
start = LADDER[0]

print(f'{"":<3}{"what we tried":<42}{"wall match":>11}{"vs start":>10}{"rooms":>8}')
print("-" * 74)
for i, e in enumerate(LADDER):
    delta = e["score"] - start["score"]
    change = "—" if i == 0 else f'{delta:+.0%}'
    mark = " ←" if e is best else ""
    print(f'{i:<3}{e["name"][:41]:<42}{e["score"]:>10.0%}{change:>10}{e["rooms"]:>8}{mark}')
print("-" * 74)
print(f'\nbest: {best["name"]} — {best["score"]:.0%}, '
      f'up from {start["score"]:.0%} where we started')
print(f'({best["note"]})')

### The chart

In [ ]:
import matplotlib.pyplot as plt

# One measure, one series, so one colour for every bar — the length is the message
# and a second colour would just repeat it. The best bar is called out with a label,
# and the dashed line is where we started.
BAR = "#2a78d6"
INK, MUTED, RULE = "#0b0b0b", "#52514e", "#d8d8d2"

names = [e["name"] for e in LADDER]
scores = [e["score"] for e in LADDER]
y = np.arange(len(LADDER))

fig, ax = plt.subplots(figsize=(10, 0.62 * len(LADDER) + 1.9))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

ax.barh(y, scores, height=0.6, color=BAR, zorder=3)
ax.axvline(scores[0], color=MUTED, linestyle="--", linewidth=1, zorder=2)
ax.text(scores[0], -0.72, "where we started", color=MUTED, fontsize=9,
        ha="center", va="bottom")

for i, s in enumerate(scores):
    label = f"{s:.0%}" + ("   best" if LADDER[i] is best else "")
    ax.text(s + 0.012, i, label, va="center", fontsize=10,
            color=INK, fontweight="bold" if LADDER[i] is best else "normal")

ax.set_yticks(y)
ax.set_yticklabels(names, fontsize=10, color=INK)
ax.invert_yaxis()
ax.set_xlim(0, max(scores) * 1.22)
ax.set_xticks(np.arange(0, 1.01, 0.2))
ax.set_xticklabels([f"{v:.0%}" for v in np.arange(0, 1.01, 0.2)], color=MUTED, fontsize=9)
ax.set_xlabel("share of every room's edge that lands on a wall", color=MUTED, fontsize=10)
ax.grid(axis="x", color=RULE, linewidth=1, zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(RULE)
ax.tick_params(length=0)
ax.set_title(f"Every change we tried, measured on the same {LADDER[0]['plans']} plans",
             fontsize=13, color=INK, loc="left", pad=22)
plt.tight_layout()
plt.show()

### Every flat: ours, theirs, and the two together

Four panels per plan.

| panel | what it is |
|---|---|
| 1 | the plan, untouched |
| 2 | **ours** — the wall model, rooms grown from the printed room names |
| 3 | **theirs** — the best the room-finder managed on its own |
| 4 | **both** — the room-finder's rooms, grown out to our walls |

Read the shapes, not the numbers. Three things to check on each:

- is an **open-plan kitchen-and-living-room one room**, or two?
- are the **bathroom, WC, hallway and cupboards** there at all?
- do the edges **run along the walls** — and do any rooms **spill outside the building**?

That last one is the blind spot in the score: a room that has ballooned out past the
outer wall still has its edge on a wall, so it can score well and look plainly wrong.
If you see it happening, trust your eyes.

In [ ]:
import colorsys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Polygon as MplPoly
from PIL import Image

READINGS = {"What we have today": ours, "Both models together": joined,
            **{k: v for k, v in SOURCES.items() if v}}

# the best the room-finder did without our help
finder_only = [e for e in LADDER if e["space"] == "original"]
finder_best = max(finder_only, key=lambda e: e["score"]) if finder_only else None
finder_reading = READINGS.get(finder_best["name"], {}) if finder_best else {}
combined = next((e for e in LADDER if e["name"] == "Both models together"), None)


def hue(i):
    return colorsys.hsv_to_rgb((i * 0.61803) % 1.0, 0.55, 0.95)


def draw(ax, img, rooms, title):
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(title, fontsize=10)
    for i, room in enumerate(rooms):
        p = np.asarray(room["polygon_px"])
        if len(p) < 3:
            continue
        c = hue(i)
        ax.add_patch(MplPoly(p, closed=True, facecolor=c + (0.35,), edgecolor=c, linewidth=2))
        if room.get("label"):
            ax.text(*p.mean(axis=0), room["label"], ha="center", va="center", fontsize=7.5,
                    weight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75))


def panel(entry, reading, lid):
    """Rooms, the picture they were drawn on, and a caption with the score."""
    rooms = (reading or {}).get(lid, {"rooms": []})["rooms"]
    img = GEOM_IMG[lid] if (entry and entry["space"] == "geometry") else ORIGINAL_IMG[lid]
    score = entry["per_plan"].get(lid) if entry else None
    caption = f"{len(rooms)} rooms"
    if score is not None:
        caption += f" · {score:.0%}"
    return img, rooms, caption


ORIGINAL_IMG = {lid: np.array(Image.open(PLANS / f"{lid}.png").convert("RGB")) for lid in IDS}

for lid in IDS:
    fig, axes = plt.subplots(1, 4, figsize=(23, 6.2))
    axes[0].imshow(ORIGINAL_IMG[lid])
    axes[0].axis("off")
    axes[0].set_title(f"{lid} — the plan", fontsize=10)

    for ax, (entry, reading, name) in zip(
            axes[1:],
            [(LADDER[0], ours, "ours"),
             (finder_best, finder_reading, "the room-finder alone"),
             (combined, joined, "both together")]):
        img, rooms, caption = panel(entry, reading, lid)
        draw(ax, img, rooms, f"{name} · {caption}")
    plt.tight_layout()
    plt.show()

### Take it home

In [ ]:
import json
import shutil
from pathlib import Path

OUT = ROOT / "results"
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir()

(OUT / "ladder.json").write_text(json.dumps(
    [{k: v for k, v in e.items()} for e in LADDER], indent=1))
for name, reading in READINGS.items():
    slug = name.lower().replace(" ", "_").replace(",", "").replace("-", "_")
    (OUT / f"{slug}.json").write_text(json.dumps(reading, indent=1))

archive = shutil.make_archive(str(ROOT / "results"), "zip", OUT)
print(f"written to  {archive}")
print()
print("Open the file browser in the sidebar, find results.zip, and download it.")
print("It holds every reading and the score for every plan at every step.")

---

## What this means

**If "Both models together" won**, that is the answer, and it is the one to wire into the
pipeline. Nothing needs training and nothing needs labelling: the room-finder and the wall
model are both already downloaded, and the growing-outwards part is code we already run.
Drop the zip in the repo and say so.

**If an earlier step won**, take that one — the table says exactly which and by how much.

**If nothing beat the starting point**, that is a real answer too, and a useful one. It
means the room-finder is not seeing our plans well enough for its rooms to be worth growing,
and the next thing to try is teaching it on our own plans rather than tuning what it already
does.

Whatever the numbers say, **the pictures decide**. A score that goes up while the pictures
get worse means the score is measuring the wrong thing — that has happened once already on
this project, and it is worth half a minute of scrolling to rule out.